## Silver – transaction_items

#### Purpose

Transform the Bronze `transaction_items` table into a clean and analytics-ready Silver table by:

- Standardizing column names using the reusable UDF    
- Quarantining malformed / invalid records into a quarantine table 
- Rolling up exact duplicate line-items  
  - If multiple rows are exactly the same (same `transaction_id`, `item_id`, `unit_price`, `created_at`)  
  - then we aggregate them into a single record by:
    - summing `quantity`
    - summing `subtotal`
- Generating a deterministic surrogate key (`transaction_item_sk`)  
  - Since this dataset does not contain a natural primary key  
  - We create a stable hash-based key so the same row always produces the same key

#### Source

- `coffee.bronze.transaction_items`

#### Targets

- `coffee.silver.transaction_items`  
- `{catalog}.{silver_schema}.{source_table}_quarantine`


#### Key Notes

- A true natural primary key does not exist in the raw dataset for transaction items.  
- Composite keys like `(transaction_id, item_id, created_at)` were not reliable  because some transactions contain repeated identical items.  
- Therefore we implemented:
  - roll-up aggregation to handle exact duplicates correctly  
  - a surrogate key for uniqueness and downstream joins


In [0]:
%python
# Widgets allow the same notebook to be executed across environments (DEV/PROD)
# and reused across multiple tables by changing only job parameters.
#
# default_watermark is used when the Silver table is empty (first run),
# enabling incremental ingestion logic without special casing.

dbutils.widgets.text("catalog", "coffee")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("source_table", "transaction_items")
dbutils.widgets.text("default_watermark", "1900-01-01")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
source_table = dbutils.widgets.get("source_table")
default_watermark = dbutils.widgets.get("default_watermark")


In [0]:
%run ./Silver_utils/silver_transform_utils

In [0]:
%python
df_bronze = spark.table(f"{catalog}.{bronze_schema}.{source_table}")


In [0]:
%python
# Standardize all incoming column names using the central UDF.
# This ensures consistent snake_case naming across all Silver tables,
# regardless of how the raw files were named in Bronze.

df_std = standardize_columns(df_bronze)


In [0]:
%python
df_std.createOrReplaceTempView(f"bronze_{source_table}_std")


In [0]:
%python
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{silver_schema}.{source_table} (

  transaction_item_sk STRING,
  transaction_id STRING,
  item_id INT,
  quantity INT,
  unit_price DOUBLE,
  subtotal DOUBLE,
  created_at TIMESTAMP,
  row_count INT,

  -- Bronze metadata
  loaded_at TIMESTAMP,
  updated_at TIMESTAMP,
  load_dt DATE,
  source_file STRING,

  -- Silver audit
  silver_loaded_at TIMESTAMP,
  silver_updated_at TIMESTAMP
)
USING DELTA
""")


In [0]:
-- Incremental extraction:
-- Only process Bronze rows that arrived after the latest loaded_at timestamp
-- already present in the Silver target table.
--
-- This prevents reprocessing old Bronze records and keeps Silver rerun-safe.

CREATE OR REPLACE TEMP VIEW bronze_transaction_items_incremental AS
SELECT *
FROM bronze_transaction_items_std
WHERE loaded_at >
(
  SELECT COALESCE(MAX(loaded_at), '1900-01-01')
  FROM coffee.silver.transaction_items
);


In [0]:
%python

# ---------------------------------------------
# We first count rows that violate Silver rules.
# If the count is 0, we skip quarantine table creation completely.

invalid_count = spark.sql("""
SELECT COUNT(*) AS cnt
FROM bronze_transaction_items_incremental
WHERE
  transaction_id IS NULL
  OR item_id IS NULL
  OR quantity IS NULL
  OR unit_price IS NULL
  OR subtotal IS NULL
  OR created_at IS NULL
""").collect()[0]["cnt"]

print("Invalid rows:", invalid_count)


In [0]:
%python

 #Create and load quarantine table ONLY if needed

# If invalid_count > 0:
#   1) Create quarantine table (if not already created)
#   2) Merge bad records into it
# Else:
#   Skip quarantine creation entirely

if invalid_count > 0:

  
    #Create quarantine table (only when needed)
  
    # This table stores bad records for audit/debugging.
    # We keep raw types as STRING for flexibility.
    # We also add quarantine_reason + quarantined_at.

    spark.sql("""
    CREATE TABLE IF NOT EXISTS {catalog}.{silver_schema}.{source_table}_quarantine (
      transaction_id STRING,
      item_id STRING,
      quantity STRING,
      unit_price STRING,
      subtotal STRING,
      created_at STRING,

      -- Bronze metadata
      loaded_at TIMESTAMP,
      updated_at TIMESTAMP,
      load_dt DATE,
      source_file STRING,

      -- Quarantine metadata
      quarantine_reason STRING,
      quarantined_at TIMESTAMP
    )
    USING DELTA
    """)

  
    #  Insert bad rows using MERGE (idempotent)
  
    # MERGE is used so reruns do not insert duplicates
    # into the quarantine table.
    #
    # Matching logic:
    # - Same transaction_id + item_id + quarantine_reason
    #
    # Note:
    # We use INSERT * because the source query returns
    # all base columns + quarantine_reason + quarantined_at.

    spark.sql("""
    MERGE INTO {catalog}.{silver_schema}.{source_table}_quarantine q
    USING (
      SELECT
        *,
        CASE
          WHEN transaction_id IS NULL THEN 'transaction_id is null'
          WHEN item_id IS NULL THEN 'item_id is null'
          WHEN quantity IS NULL THEN 'quantity is null'
          WHEN unit_price IS NULL THEN 'unit_price is null'
          WHEN subtotal IS NULL THEN 'subtotal is null'
          WHEN created_at IS NULL THEN 'created_at is null'
          ELSE 'unknown validation failure'
        END AS quarantine_reason,
        current_timestamp() AS quarantined_at
      FROM bronze_transaction_items_incremental
      WHERE
        transaction_id IS NULL
        OR item_id IS NULL
        OR quantity IS NULL
        OR unit_price IS NULL
        OR subtotal IS NULL
        OR created_at IS NULL
    ) b
    ON q.transaction_id = b.transaction_id
    AND q.item_id = b.item_id
    AND q.quarantine_reason = b.quarantine_reason
    WHEN NOT MATCHED THEN
    INSERT *;
    """)

else:
  
    # If no invalid records exist, skip quarantine logic
  
    print("No invalid rows found. Quarantine table not created.")


In [0]:
%python
spark.sql(f""" 
MERGE INTO  {catalog}.{silver_schema}.{source_table} s
USING (

 
  -- Step 1: Filter valid rows + rollup duplicates
 
  SELECT
    transaction_id,
    CAST(item_id AS INT) AS item_id,

    -- Rollup duplicates
    SUM(CAST(quantity AS INT)) AS quantity,
    CAST(unit_price AS DOUBLE) AS unit_price,
    SUM(CAST(subtotal AS DOUBLE)) AS subtotal,

    CAST(created_at AS TIMESTAMP) AS created_at,

    -- Bronze metadata (keep latest)
    MAX(loaded_at) AS loaded_at,
    MAX(updated_at) AS updated_at,
    MAX(load_dt) AS load_dt,
    ANY_VALUE(source_file) AS source_file,

   
    -- Step 2: Deterministic surrogate key
   
    sha2(
      concat_ws('|',
        transaction_id,
        CAST(item_id AS STRING),
        CAST(unit_price AS STRING),
        CAST(created_at AS STRING)
      ),
      256
    ) AS transaction_item_sk,

    -- How many rows were rolled up
    COUNT(*) AS row_count

  FROM bronze_transaction_items_incremental b
  WHERE
    transaction_id IS NOT NULL
    AND item_id IS NOT NULL
    AND quantity IS NOT NULL
    AND unit_price IS NOT NULL
    AND subtotal IS NOT NULL
    AND created_at IS NOT NULL

  GROUP BY
    transaction_id,
    item_id,
    unit_price,
    created_at

) b


-- Step 3: Match on surrogate key (stable primary key)

ON s.transaction_item_sk = b.transaction_item_sk


-- Step 4: If same SK exists → update values

WHEN MATCHED THEN
UPDATE SET
  s.quantity          = b.quantity,
  s.subtotal          = b.subtotal,

  -- Bronze metadata refresh
  s.loaded_at         = b.loaded_at,
  s.updated_at        = b.updated_at,
  s.load_dt           = b.load_dt,
  s.source_file       = b.source_file,

  -- Rollup metadata refresh
  s.row_count         = b.row_count,

  -- Silver audit
  s.silver_updated_at = current_timestamp()


-- Step 5: If new SK → insert

WHEN NOT MATCHED THEN
INSERT (
  transaction_id,
  item_id,
  quantity,
  unit_price,
  subtotal,
  created_at,
  loaded_at,
  updated_at,
  load_dt,
  source_file,
  transaction_item_sk,
  row_count,
  silver_loaded_at,
  silver_updated_at
)
VALUES (
  b.transaction_id,
  b.item_id,
  b.quantity,
  b.unit_price,
  b.subtotal,
  b.created_at,
  b.loaded_at,
  b.updated_at,
  b.load_dt,
  b.source_file,
  b.transaction_item_sk,
  b.row_count,
  current_timestamp(),
  current_timestamp()
)
""")
